# Model template

**Copy this notebook into a new folder under `notebooks/` and rename it.**

Read [docs/getting-started-as-a-modeler.md](../docs/getting-started-as-a-modeler.md)
first if you have not already.

## How to use this template

The five sections below are in this order for a reason. Sections 2 and 3 are
the strict ones:

| Section | Rule |
|---|---|
| 1. Base data | Load it. Never fetch your own copy. |
| 2. Parameters | **Every** knob in one cell. Nothing inline. |
| 3. The rule | One pure function. No globals, no in-place mutation. |
| 4. Validation | Distribution, spot checks, and a diff against the published score. |
| 5. Promotion | What has to be true before this becomes real. |

Sections 2 and 3 are what make a model *promotable*. If the group adopts your
model, the parameters cell becomes a section of `config.py` and the pure
function becomes a module under `models/` — **copied, not rewritten**.
Reimplementation is where scores quietly change.

Delete this cell when you copy the notebook.

---
# 0. Setup

In [ ]:
import geopandas as gpd
import pandas as pd

# The published model is importable. Use it rather than retyping its numbers —
# they will go stale the moment somebody tunes them.
from ridescore import config
from ridescore.models.lts import lts_level

pd.set_option("display.max_columns", 50)

---
# 1. Load the base data

Every model scores the same segments. See
[BaseData/](BaseData/README.md) for what the attributes mean — especially
which ones are *filled in* rather than recorded.

> **Not yet available.** `ridescore fetch` is still being ported from
> `notebooks/archive/data_processing.ipynb`. Until it lands, follow that
> notebook's cells 2–6, or ask in `#ridescore-dc` for a current extract.

In [ ]:
# TODO: replace with the shared loader once `ridescore fetch` lands.
#   uv run ridescore fetch --run-date 2026-08-10
#   segments = gpd.read_parquet("raw/2026-08-10/segments.parquet")

SEGMENTS_PATH = "../BaseData/segments.parquet"   # adjust to where yours lives

segments = gpd.read_parquet(SEGMENTS_PATH)
print(f"{len(segments):,} segments, CRS {segments.crs}")
segments.head(3)

In [ ]:
# Before you model anything: how much of each column is actually populated?
# A column that is 70% imputed will not support the conclusion you want it to.
(
    segments.notna().mean().sort_values()
    .to_frame("populated")
    .style.format("{:.1%}")
)

---
# 2. Parameters

**Every threshold, weight and lookup table you use goes in the cell below.**
Nothing inline, nothing buried inside a function.

If your model is adopted, this cell becomes your model's OWN
`models/<your_model>/config.py` — you never edit a shared settings file, so you
cannot collide with another model's `W_CRASH`. Treat it as the thing a reviewer
reads to understand your model in one screen.

Say *why* a value is what it is. `SPEED_THRESHOLD = 25` tells a reviewer
nothing; a one-line comment saying where 25 came from tells them everything.

In [ ]:
# --- what counts as a stressful street ------------------------------------
# 25 mph is DC's default residential limit, so it separates "posted as
# residential" from "posted as something else".
MY_SPEED_THRESHOLD = 25
MY_LANES_THRESHOLD = 2

# --- component scores, 0-100 ----------------------------------------------
MY_FACILITY_TO_SCORE = {
    "protected_track": 100,
    "buffered_lane": 75,
    "painted_lane": 50,
    "none": 0,
}
MY_FACILITY_DEFAULT = 0

# --- the blend -------------------------------------------------------------
# Must sum to 1.0 — asserted below rather than trusted.
MY_W_STRESS = 0.7
MY_W_FACILITY = 0.3

assert abs((MY_W_STRESS + MY_W_FACILITY) - 1.0) < 1e-9, "weights must sum to 1"

---
# 3. The rule

**One pure function: `f(attributes) -> score`.**

- takes everything it needs as arguments
- reads no globals except the parameters above
- returns a value; mutates nothing
- no `gdf` in sight

Pure means testable, and testable means promotable. This function becomes a
module under
[`models/`](../tools/ridescore-cli/src/ridescore/models/) unchanged.

In [ ]:
def my_model_score(facility: str, speed: float, lanes: float, function: str = "") -> float:
    """Score one segment, 0 (hostile) to 100 (calm).

    `speed` and `lanes` are the FILLED values, not the raw ones — a missing
    lane count has already become 1 and a missing speed limit 25 by the time
    this is called. That is a real modelling assumption; see BaseData/README.md.
    """
    stress = 100.0 if (speed <= MY_SPEED_THRESHOLD and lanes <= MY_LANES_THRESHOLD) else 25.0
    facility_score = MY_FACILITY_TO_SCORE.get(facility, MY_FACILITY_DEFAULT)
    return MY_W_STRESS * stress + MY_W_FACILITY * facility_score

In [ ]:
# Applying it is the only place a dataframe is touched. Keep it to one cell so
# the rule above stays independent of how it happens to be applied.
segments["my_model_score"] = segments.apply(
    lambda r: my_model_score(
        r.bike_facility_type, float(r.speed_limit), float(r.num_lanes), r.function
    ),
    axis=1,
)
segments["my_model_score"].describe().round(1)

---
# 4. Validation

Four questions, in order of how convincing the answers are.

### 4a. What does the distribution look like?

A score that clusters on one value is not distinguishing anything. A score
that is bimodal usually means one input is dominating.

In [ ]:
ax = segments["my_model_score"].plot.hist(bins=40, figsize=(8, 3))
ax.set_xlabel("my_model_score")
ax.set_title(f"n = {len(segments):,}")

### 4b. Does it agree with streets you actually know?

Pick five or six you have ridden — one you know is awful, one you know is
pleasant, and some in between. If the model disagrees with your own experience,
find out why before going further.

In [ ]:
KNOWN_STREETS = [
    # ("street name", what you expect: "low" | "mid" | "high"),
    ("15TH ST NW", "high"),        # protected two-way track
    ("RHODE ISLAND AVE NE", "low"),  # wide, fast, no facility
]

for name, expected in KNOWN_STREETS:
    hit = segments[segments["route_name"].str.upper() == name.upper()]
    if hit.empty:
        print(f"{name}: NOT FOUND — check the exact route_name spelling")
        continue
    print(f"{name}: {hit['my_model_score'].mean():.1f}  (expected {expected})")

### 4c. Where do you disagree with the published score?

**This is the most valuable output in the notebook.** The streets where your
model and `ridescore_v1` differ — and your explanation of why you are right —
are what a reviewer will actually engage with.

A model that agrees everywhere adds nothing. A model that disagrees everywhere
probably has a bug.

In [ ]:
comparison = segments[["route_name", "bike_facility_type", "speed_limit", "num_lanes"]].copy()
comparison["published"] = segments["ridescore_v1"]
comparison["mine"] = segments["my_model_score"]
comparison["delta"] = comparison["mine"] - comparison["published"]

print(f"agree within 5 points: {(comparison['delta'].abs() <= 5).mean():.1%}")
print("\nWhere I score a street MUCH better than the published score:")
display(comparison.nlargest(10, "delta"))
print("\nWhere I score it MUCH worse:")
display(comparison.nsmallest(10, "delta"))

### 4d. What did you not validate?

Write it down here, honestly. "I could not work out how to handle one-way
pairs" is genuinely useful to a reviewer. A confident model with a quiet gap in
it is not.

_Replace this cell with what you are unsure about._

- ...
- ...

---
# 5. Promotion checklist

Only relevant if the group adopts your model. Nothing below happens without
that ([level 3](../docs/ongoing-data-pipeline.md)).

- [ ] Section 2 can be pasted into `config.py` **unchanged**
- [ ] Section 3 can be pasted into `models/<your_model>/` **unchanged**
- [ ] Every threshold in section 2 has a test at the value that passes and the
      value that does not — see
      [`test_lts.py`](../tools/ridescore-cli/tests/models/lts/test_rules.py)
- [ ] The 4c comparison is explained, not just displayed
- [ ] Your folder's `README.md` says what you are unsure about
- [ ] Nothing here reaches the network except through the shared base data
- [ ] You have posted in `#ridescore-dc` and opened a PR into `develop` from a
      `feature/...` branch

If you find a bug in the published model while doing this: **promote yours
faithfully first, and fix the bug as its own separate change.** A change that
does two things at once cannot be verified against either.